# Petti Vision — Colab training
GPU runtime сонгоно уу (Runtime → Change runtime type → T4/L4/A100).

1. Public dataset татах → 2. Feature cache → 3. Attribute head сургах → 4. Re-ID → 5. Evaluate → 6. artifacts татаж авах

In [ ]:
!nvidia-smi -L
!git clone https://github.com/GankhulugD/petti-vision-test.git
%cd petti-vision-test/apps/model
!pip -q install -r requirements.txt

### (Сонголттой) Hugging Face token — DINOv3 гэх мэт gated модел ашиглах бол
`PETTI_REID_MODEL=facebook/dinov3-vits16-pretrain-lvd1689m`

In [ ]:
import os
# os.environ['HF_TOKEN'] = '...'
# os.environ['PETTI_VLM_MODEL'] = 'google/siglip2-so400m-patch14-384'  # хамгийн нарийвчлалтай
os.environ['PETTI_DEVICE'] = 'cuda'

## 1. Өгөгдөл
Oxford-IIIT Pet (37 үүлдэр) + Stanford Dogs (120 үүлдэр). Өөрийн зураг: `data/my_photos/<breed_key>/<pet_id>/*.jpg` хэлбэрээр Drive-аас хуулаад `--folder` ашиглана.

In [ ]:
!python3 training/prepare_data.py --oxford --stanford-dogs
# from google.colab import drive; drive.mount('/content/drive')
# !cp -r /content/drive/MyDrive/petti_photos data/my_photos && python3 training/prepare_data.py --folder data/my_photos

## 2-3. Feature cache + attribute heads
Эхний удаа backbone-оор feature гаргана (~10-15 мин T4 дээр), дараа нь head сургалт 1-2 минут.

In [ ]:
!python3 training/train_heads.py --epochs 60
!cat artifacts/heads_report.txt

## 4. Re-ID (ижил амьтан мөн эсэх)
Бүртгэлийн өгөгдөл (`export_registry.py`) эсвэл PetFace зэрэг identity dataset хэрэгтэй (`<species>/<identity>/*.jpg`).

In [ ]:
# !python3 training/export_registry.py
# !python3 training/train_reid.py --epochs 30

## 5. Evaluate + calibration

In [ ]:
# pet_id-тай manifest шаардлагатай
# !python3 training/evaluate.py --manifest data/manifests/registry.csv

## 6. Artifacts татах → `apps/model/artifacts/` руу хуулна

In [ ]:
!cd artifacts && zip -q ../artifacts.zip heads.npz heads.json heads_report.txt $(ls reid_projection.* calibration.json 2>/dev/null)
from google.colab import files; files.download('artifacts.zip')